# 00 Export Metadata: item titles + images for the app MVP

Produces `DATA_DIR/artifacts/items_metadata.parquet` (`item_id, domain, title,
image_url`) for the items already known to the trained SVD/ALS/item2vec/SASRec/BERT4Rec
models (Task 2 export cells in the `svd`/`als`/`item-2-vec`/`transformer` branches
must run first -- this notebook reads their `item2idx.json` / `word2vec.model`).

Colab-only: not executed as part of this change. `images` schema on the
McAuley-Lab/Amazon-Reviews-2023 HF dataset is not verified here -- see the
warning before `extract_image_url`.

## Setup

In [1]:
!test -d /content/recommendation-system/.git \
  || git clone -b app https://github.com/mkobycheva/recommendation-system.git /content/recommendation-system

%cd /content/recommendation-system
!git fetch origin app
!git checkout app
!git reset --hard origin/app
!git log -1 --oneline
!pip install -r requirements.txt -q
!pip install datasets -q

import sys
sys.path.insert(0, '.')

from google.colab import drive
drive.mount('/content/drive')


/content/recommendation-system
From https://github.com/mkobycheva/recommendation-system
 * branch            app        -> FETCH_HEAD
Already on 'app'
Your branch is up to date with 'origin/app'.
HEAD is now at 1327c11 Cap numpy at <2.1 to avoid breaking Colab's bundled numba
1327c11 (HEAD -> app, origin/app) Cap numpy at <2.1 to avoid breaking Colab's bundled numba
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Imports

In [2]:
import json
import os

import pandas as pd
from datasets import load_dataset
from gensim.models import Word2Vec


## Load known items from each trained model

`item2vec` doesn't export its own `item2idx.json` (Task 2) -- its
`word2vec.model` is self-describing, `model.wv.index_to_key` *is* its
vocabulary, so we read that directly instead. `svd`/`als`/`sasrec`/`bert4rec`
all export `item2idx.json` with the same `add_domain_item_ids` convention
(`{domain}::{parent_asin}`).

In [3]:
DATA_DIR = '/content/drive/MyDrive/recsys-data'
ARTIFACTS_DIR = f'{DATA_DIR}/artifacts'


def load_known_items(model_name):
    if model_name == 'item2vec':
        model = Word2Vec.load(f'{ARTIFACTS_DIR}/item2vec/word2vec.model')
        return set(model.wv.index_to_key)

    with open(f'{ARTIFACTS_DIR}/{model_name}/item2idx.json') as f:
        return set(json.load(f).keys())


# Union, not intersection of the five models with each other: we want any
# item that at least one model knows, so it's not "no model knows this title".
known_items = set()
for model_name in ('svd', 'als', 'item2vec', 'sasrec', 'bert4rec'):
    known_items |= load_known_items(model_name)

needed_asins = {
    'books': {item_id.split('::', 1)[1] for item_id in known_items if item_id.startswith('books::')},
    'movies': {item_id.split('::', 1)[1] for item_id in known_items if item_id.startswith('movies::')},
}
print({domain: len(asins) for domain, asins in needed_asins.items()})


{'books': 404630, 'movies': 182434}


## Stream-filter the Amazon Reviews 2023 metadata catalog

`raw_meta_Books`/`raw_meta_Movies_and_TV` are the full catalog (millions of
rows, several GB) -- we only need the `needed_asins` intersection (thousands
of items). `streaming=True` reads row-by-row from the network and discards
non-matches immediately; nothing from the raw catalog is cached to disk.

In [ ]:
CATEGORY_CONFIG = {
    'books': 'raw_meta_Books',
    'movies': 'raw_meta_Movies_and_TV',
}


def collect_metadata(config_name, needed):
    # datasets>=4.0 dropped support for loading scripts / trust_remote_code
    # entirely (hard removal, not a deprecation) -- this repo still has an
    # Amazon-Reviews-2023.py script, so the plain call fails now. Read from
    # the auto-converted parquet revision instead (no script execution).
    # If this raises "BuilderConfig ... not found", the config name didn't
    # survive the auto-conversion -- list what's actually there with:
    #   from huggingface_hub import HfApi
    #   HfApi().list_repo_files('McAuley-Lab/Amazon-Reviews-2023', repo_type='dataset', revision='refs/convert/parquet')
    stream = load_dataset(
        'McAuley-Lab/Amazon-Reviews-2023', config_name,
        split='full', streaming=True,
        revision='refs/convert/parquet',
    )
    matched = []
    for row in stream:
        if row['parent_asin'] in needed:
            matched.append(row)
            if len(matched) == len(needed):  # early stop once everything is found
                break
    return matched


matched_rows = {}
for domain, config_name in CATEGORY_CONFIG.items():
    print(f'Streaming {config_name}, looking for {len(needed_asins[domain]):,} items...')
    matched_rows[domain] = collect_metadata(config_name, needed_asins[domain])
    print(f'  found {len(matched_rows[domain]):,} / {len(needed_asins[domain]):,}')


## Save the filtered raw rows locally (separate from the final parquet)

The expensive pass (streaming millions of rows) happens once here. Parsing
`title`/`images` below will likely need a few iterations to match the real
schema -- that parsing step reads this small local file, not the network.

In [ ]:
os.makedirs(ARTIFACTS_DIR, exist_ok=True)

for domain, rows in matched_rows.items():
    path = f'{ARTIFACTS_DIR}/matched_{domain}_raw.jsonl'
    with open(path, 'w') as f:
        for row in rows:
            f.write(json.dumps(row, default=str) + '\n')
    print(f'Saved {len(rows):,} rows to {path}')


### Non-streaming alternative (not used here)

If this ever needs to be rerun for a different `needed_asins` and Colab disk
space allows, the non-streaming form caches the raw catalog locally so
reruns don't re-download from the network:

```python
ds = load_dataset('McAuley-Lab/Amazon-Reviews-2023', 'raw_meta_Books', split='full', revision='refs/convert/parquet')
ds = ds.filter(lambda r: r['parent_asin'] in needed_asins['books'], num_proc=4)
```

Not used above since this notebook only needs to run once per model refresh.

## Parse title + image

⚠️ **Schema not verified without running this notebook.** `raw_meta_*` on HF
typically has `parent_asin`, `title`, `images`, among others, but the exact
shape of `images` (dict-of-lists from Arrow conversion vs. list-of-dict from
raw jsonl vs. plain URL list/string) needs to be checked against
`matched_rows['books'][0]` the first time this actually runs, and
`extract_image_url` adjusted if none of the branches below match.

In [ ]:
def extract_image_url(row):
    images = row.get('images')
    if not images:
        return None

    # Most likely on HF: dict-of-lists (Arrow-friendly columnar form), e.g.
    # {"hi_res": [...], "large": [...], "thumb": [...], "variant": [...]}.
    if isinstance(images, dict):
        for key in ('hi_res', 'large', 'thumb'):
            values = images.get(key)
            if values:
                first = next((v for v in values if v), None)
                if first:
                    return first

    # Fallback: raw-jsonl style list of per-image dicts.
    if isinstance(images, list) and images and isinstance(images[0], dict):
        for item in images:
            for key in ('hi_res', 'large', 'thumb'):
                if item.get(key):
                    return item[key]

    # Fallback: plain list of URL strings, or a single URL string.
    if isinstance(images, list) and images and isinstance(images[0], str):
        return images[0]
    if isinstance(images, str):
        return images

    return None


In [ ]:
def parse_rows(rows, domain):
    parsed = []
    for row in rows:
        parent_asin = row.get('parent_asin')
        if not parent_asin:
            continue
        parsed.append({
            'item_id': f'{domain}::{parent_asin}',
            'domain': domain,
            'title': row.get('title') or '',
            'image_url': extract_image_url(row),
        })
    return parsed


items_metadata_rows = []
for domain, rows in matched_rows.items():
    items_metadata_rows.extend(parse_rows(rows, domain))

items_metadata = pd.DataFrame(items_metadata_rows, columns=['item_id', 'domain', 'title', 'image_url'])
items_metadata.head()


## Save `items_metadata.parquet`

In [ ]:
output_path = f'{ARTIFACTS_DIR}/items_metadata.parquet'
items_metadata.to_parquet(output_path, index=False)
print(f'Saved {len(items_metadata):,} rows to {output_path}')
